# Create a web app URL to visit a given DSS object handle

In [0]:
# some syntax requires python 3.10+
import sys
assert sys.version_info >= (3, 10), "Python 3.10 or higher is required"

import dataiku


In [0]:
# copy and paste from xzibit.utils.py
import os
def get_dss_external_url():
    """TBD"""
    # 1. Initialize the client (connects to local instance)
    client = dataiku.api_client()

    # 2. Retrieve General Settings (Requires Admin permissions)
    # This corresponds to the "Administration > Settings > General" page
    settings = client.get_general_settings()

    # 3. Extract the 'studioExternalUrl' from the raw settings dictionary
    # This key holds the value of the "DSS URL" field
    dss_url = settings.get_raw().get("studioExternalUrl")

    if dss_url:
        return dss_url

    return None


def get_dss_url_from_env():
    """TBD"""
    # Attempt to retrieve host and port from environment variables
    ext_host = os.environ.get("DKU_BACKEND_EXT_HOST")
    base_port = os.environ.get("DKU_BASE_PORT")

    if ext_host and base_port:
        # Note: You may need to infer the scheme (http vs https)
        # based on your knowledge of the instance setup.
        return f"http://{ext_host}:{base_port}"
    return None


def get_dss_url_from_global_vars():
    """TBD"""
    client = dataiku.api_client()

    # Retrieve global variables (accessible to all users)
    global_vars = client.get_variables()

    # Check for common naming conventions like 'dss_url', 'public_url', or 'instance_url'
    return global_vars.get("dss_url") or global_vars.get("public_url")


def get_dss_base_url():
    """returns the base URL for the local node, without a trailing slash"""
    res = (get_dss_url_from_env()
        or get_dss_external_url()
        or get_dss_url_from_global_vars() )
    return res.rstrip('/')


In [0]:
# new custom functions

def get_dss_object_type(obj_handle):
    """x"""
    return obj_handle.__class__.__name__


def get_dss_object_url(object_handle, project_key=None, object_id=None):
    """TBD"""
    baseurl = get_dss_base_url()

    # at least one is None, return None
    if any(v is None for v in (baseurl, object_handle)):
        return None
    
    class_name = get_dss_object_type(object_handle)

    match class_name:
        case "DSSProject":
            return f"{baseurl}/projects/{project_key}/flow/"
        case "DSSAgentToolListItem":
            return f"{baseurl}/projects/{project_key}/agent-tools/{object_handle.id}"
        case "DSSCluster":
            return f"{baseurl}/admin/clusters/{object_id}"
        case _:
            x = str(type(object_handle))
            return f"ERROR: {x}"


In [0]:
client = dataiku.api_client()

for cluster in client.list_clusters():
    cluster_id = cluster.get("id", None)
    cluster_handle = client.get_cluster(cluster_id)
    res_url = get_dss_object_url(cluster_handle, object_id=cluster_id)
    print(f"cluster: {res_url}")

for code_env in client.list_code_envs():
    
    
    
for project_key in client.list_project_keys():
    project = client.get_project(project_key)
    res_url = get_dss_object_url(project, project_key=project_key)
    print(f"project: {res_url}")

    
    for agent_item in project.list_agent_tools():
        res_url = get_dss_object_url(agent_item, project_key=project_key)
        print(f"\tagent_tool: {res_url}")
